In [1]:
!pip install -q transformers torch
from transformers import pipeline

# device=0 means use the GPU
clf = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, device=0)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()  # click the button and choose news_5_stocks_articles.csv
news = pd.read_csv("news_5_stocks_articles.csv", dtype=str)
texts = news["Article"].fillna("").tolist()
print(len(texts), "articles loaded")

Saving news_5_stocks_articles.csv to news_5_stocks_articles.csv
37934 articles loaded


In [3]:
rows = []
for result in clf(texts, batch_size=16, truncation=True, max_length=512):
    rows.append({x["label"].lower(): x["score"] for x in result})

scores = pd.DataFrame(rows).rename(columns=lambda c: f"p_{c}")
out = pd.concat([news.reset_index(drop=True), scores], axis=1)
out["sentiment"] = out["p_positive"] - out["p_negative"]
print(len(out), "rows scored")

37934 rows scored


In [4]:
# Drop the long article text before saving, so the file stays small
out.drop(columns=["Article"]).to_csv("news_scored_articles.csv", index=False)
files.download("news_scored_articles.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>